# LightGBM

In [1]:
import h5py
import numpy as np
import lightgbm as lgb
import time

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score,
    confusion_matrix, classification_report,
    cohen_kappa_score
)
from sklearn.utils import resample


# ==========================================================
# SELECT CLASSIFICATION MODE
# ==========================================================
classification_mode = "5class"   # "binary", "3class", "5class"


# ==========================================================
# 1. LOAD DATA
# ==========================================================
hdf5_path = r"C:\Users\anita\OneDrive - Universitetet i Oslo\Masteroppgave zzz\UOslo_March2025\Combined\Step4_normalized_train_test_FINAL.h5"

with h5py.File(hdf5_path, 'r') as f:
    X_train = f['X_train'][:]
    X_test = f['X_test'][:]
    y_train = f['y_train'][:]
    y_test = f['y_test'][:]
    feature_names = f['feature_names'][:].astype(str)

print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")


# ==========================================================
# 2. FILTER VALID LABELS
# ==========================================================
valid_labels = [0,1,2,3,5]

train_mask = np.isin(y_train, valid_labels)
test_mask  = np.isin(y_test, valid_labels)

X_train = X_train[train_mask]
y_train = y_train[train_mask]
X_test  = X_test[test_mask]
y_test  = y_test[test_mask]


# ==========================================================
# 3. LABEL MAPPING
# ==========================================================

if classification_mode == "binary":

    print("\nRunning Binary Sleep vs Wake")

    sleep_stages = [1,2,3,5]

    y_train_new = np.where(np.isin(y_train, sleep_stages),0,1)
    y_test_new  = np.where(np.isin(y_test, sleep_stages),0,1)

    target_names = ["Sleep","Wake"]


elif classification_mode == "3class":

    print("\nRunning Wake vs NREM vs REM")

    y_train_new = np.copy(y_train)
    y_test_new  = np.copy(y_test)

    y_train_new[np.isin(y_train,[1,2,3])] = 1
    y_test_new[np.isin(y_test,[1,2,3])] = 1

    y_train_new[y_train == 5] = 2
    y_test_new[y_test == 5] = 2

    target_names = ["Wake","NREM","REM"]


elif classification_mode == "5class":

    print("\nRunning Full 5-Stage Classification")

    y_train_new = np.copy(y_train)
    y_test_new  = np.copy(y_test)

    target_names = ["Wake","N1","N2","N3","REM"]


# ==========================================================
# 4. FEATURE GROUPING
# ==========================================================

EEG_features = [f for f in feature_names if f.startswith(("C3","C4","A1","A2"))]
EOG_features = [f for f in feature_names if f.startswith("EOG")]
EMG_features = [f for f in feature_names if f.startswith("EMG")]
cardiac = [f for f in feature_names if f.startswith(("ECG","Pleth","Pulse","SpO2","Thorax"))]
acc = [f for f in feature_names if "Accelerometer" in f]

feature_sets = {

    "EEG": EEG_features,
    "EEG + EOG": EEG_features + EOG_features,
    "EEG + EOG + EMG": EEG_features + EOG_features + EMG_features,
    "Cardiac": cardiac,
    "Accelerometer": acc,
    "Acc + EMG": acc + EMG_features,
    "Acc + Car": acc + cardiac,
    "Acc + Car + EMG": acc + cardiac + EMG_features,
    "All PSG features": EEG_features + EOG_features + EMG_features + cardiac,
    "All": list(feature_names)
}

feature_to_idx = {f:i for i,f in enumerate(feature_names)}


# ==========================================================
# 5. MODALITY LOOP
# ==========================================================

results = {}

for name, fs in feature_sets.items():

    print("\n=======================================")
    print(f"Training with modality: {name}")
    print(f"Number of features: {len(fs)}")

    if len(fs) == 0:
        continue

    idx = [feature_to_idx[f] for f in fs]

    X_train_fs = X_train[:, idx]
    X_test_fs = X_test[:, idx]


    # ======================================================
    # BALANCING
    # ======================================================

    if classification_mode != "5class":

        unique_classes = np.unique(y_train_new)
        max_samples = max([np.sum(y_train_new == c) for c in unique_classes])

        X_bal, y_bal = [], []

        for c in unique_classes:

            X_c = X_train_fs[y_train_new == c]
            y_c = y_train_new[y_train_new == c]

            if len(X_c) < max_samples:

                X_res, y_res = resample(
                    X_c, y_c,
                    replace=True,
                    n_samples=max_samples,
                    random_state=42
                )

            else:

                X_res, y_res = X_c, y_c

            X_bal.append(X_res)
            y_bal.append(y_res)

        X_train_final = np.vstack(X_bal)
        y_train_final = np.hstack(y_bal)

    else:

        X_train_final = X_train_fs
        y_train_final = y_train_new


    # ======================================================
    # SCALING
    # ======================================================

    scaler = StandardScaler()

    X_train_final = scaler.fit_transform(X_train_final)
    X_test_fs = scaler.transform(X_test_fs)


    # ======================================================
    # MODEL
    # ======================================================

    if classification_mode == "binary":

        model = lgb.LGBMClassifier(
            objective='binary',
            class_weight='balanced',
            num_leaves=70,
            learning_rate=0.05,
            n_estimators=300,
            random_state=42,
            n_jobs=-1
            
        )

    else:

        model = lgb.LGBMClassifier(
            objective='multiclass',
            num_class=len(np.unique(y_train_new)),
            class_weight='balanced',
            num_leaves=70, #50
            learning_rate=0.1,#0.05
            n_estimators=500,#300
            random_state=42,
            n_jobs=-1
        )


    # ======================================================
    # TRAIN
    # ======================================================

    start = time.time()

    model.fit(X_train_final, y_train_final)

    print(f"Training time: {time.time()-start:.2f} sec")


    # ======================================================
    # PREDICT
    # ======================================================

    y_pred = model.predict(X_test_fs)


    # ======================================================
    # METRICS
    # ======================================================

    acc = accuracy_score(y_test_new, y_pred)
    f1 = f1_score(y_test_new, y_pred, average='macro')
    kappa = cohen_kappa_score(y_test_new, y_pred)

    cm = confusion_matrix(y_test_new, y_pred)

    #print("\nConfusion Matrix\n", cm)

    print("\nClassification Report\n")
    print(classification_report(y_test_new, y_pred, target_names=target_names))


    # ======================================================
    # STAGE-WISE SENSITIVITY & SPECIFICITY
    # ======================================================

    sensitivities = {}
    specificities = {}

    for i, label in enumerate(target_names):

        TP = cm[i,i]
        FN = np.sum(cm[i,:]) - TP
        FP = np.sum(cm[:,i]) - TP
        TN = np.sum(cm) - (TP + FN + FP)

        sens = TP / (TP + FN)
        spec = TN / (TN + FP)

        sensitivities[label] = sens
        specificities[label] = spec

        print(f"{label:10s} | Sensitivity: {sens:.3f} | Specificity: {spec:.3f}")


    results[name] = {
        "Accuracy": acc,
        "F1_macro": f1,
        "Kappa": kappa,
        "Sensitivity": sensitivities,
        "Specificity": specificities
    }


# ==========================================================
# 6. FINAL SUMMARY
# ==========================================================

print("\n\n====================== FINAL MODALITY COMPARISON ========================")
print("------------------------------------------------------------------------------")
print(f"{'Modality':20s} | {'Acc':>6s} | {'F1-macro':>8s} | {'Kappa':>6s} |")
print("-------------------------------------------------------------------------------")

for k,v in results.items():

    print(f"{k:20s} | "
          f"{v['Accuracy']:6.3f} | "
          f"{v['F1_macro']:8.3f} | "
          f"{v['Kappa']:6.3f} | "
          )

print("--------------------------------------------------------------------------------")

Train shape: (46398, 129)
Test shape: (11600, 129)

Running Full 5-Stage Classification

Training with modality: EEG
Number of features: 60
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.009698 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 15300
[LightGBM] [Info] Number of data points in the train set: 46316, number of used features: 60
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
Training time: 24.18 sec


c:\Users\anita\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



Classification Report

              precision    recall  f1-score   support

        Wake       0.82      0.90      0.85      1872
          N1       0.55      0.33      0.41       491
          N2       0.91      0.90      0.91      5546
          N3       0.90      0.90      0.90      1642
         REM       0.84      0.86      0.85      2029

    accuracy                           0.87     11580
   macro avg       0.80      0.78      0.78     11580
weighted avg       0.86      0.87      0.87     11580

Wake       | Sensitivity: 0.899 | Specificity: 0.961
N1         | Sensitivity: 0.326 | Specificity: 0.988
N2         | Sensitivity: 0.903 | Specificity: 0.917
N3         | Sensitivity: 0.900 | Specificity: 0.983
REM        | Sensitivity: 0.857 | Specificity: 0.965

Training with modality: EEG + EOG
Number of features: 84
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.007158 seconds.
You can set `force_col_wise=true` to remove the overhead.
[L

c:\Users\anita\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



Classification Report

              precision    recall  f1-score   support

        Wake       0.84      0.90      0.87      1872
          N1       0.62      0.41      0.49       491
          N2       0.92      0.92      0.92      5546
          N3       0.91      0.91      0.91      1642
         REM       0.87      0.89      0.88      2029

    accuracy                           0.89     11580
   macro avg       0.83      0.81      0.82     11580
weighted avg       0.89      0.89      0.89     11580

Wake       | Sensitivity: 0.904 | Specificity: 0.967
N1         | Sensitivity: 0.405 | Specificity: 0.989
N2         | Sensitivity: 0.918 | Specificity: 0.927
N3         | Sensitivity: 0.911 | Specificity: 0.985
REM        | Sensitivity: 0.894 | Specificity: 0.972

Training with modality: EEG + EOG + EMG
Number of features: 87
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.007653 seconds.
You can set `force_col_wise=true` to remove the overhe

...


Classification Report

              precision    recall  f1-score   support

        Wake       0.80      0.87      0.83      1872
          N1       0.57      0.36      0.44       491
          N2       0.88      0.88      0.88      5546
          N3       0.89      0.87      0.88      1642
         REM       0.82      0.83      0.83      2029

    accuracy                           0.85     11580
   macro avg       0.79      0.76      0.77     11580
weighted avg       0.84      0.85      0.84     11580

Wake       | Sensitivity: 0.873 | Specificity: 0.958
N1         | Sensitivity: 0.360 | Specificity: 0.988
N2         | Sensitivity: 0.881 | Specificity: 0.891
N3         | Sensitivity: 0.867 | Specificity: 0.981
REM        | Sensitivity: 0.834 | Specificity: 0.960

Training with modality: All PSG features
Number of features: 103
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.008710 seconds.
You can set `force_col_wise=true` to remove the over

c:\Users\anita\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



Classification Report

              precision    recall  f1-score   support

        Wake       0.85      0.91      0.88      1872
          N1       0.63      0.40      0.49       491
          N2       0.92      0.92      0.92      5546
          N3       0.91      0.91      0.91      1642
         REM       0.87      0.90      0.89      2029

    accuracy                           0.89     11580
   macro avg       0.84      0.81      0.82     11580
weighted avg       0.89      0.89      0.89     11580

Wake       | Sensitivity: 0.909 | Specificity: 0.969
N1         | Sensitivity: 0.399 | Specificity: 0.989
N2         | Sensitivity: 0.920 | Specificity: 0.927
N3         | Sensitivity: 0.907 | Specificity: 0.985
REM        | Sensitivity: 0.898 | Specificity: 0.972

Training with modality: All
Number of features: 121
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.013346 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightG

c:\Users\anita\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



Classification Report

              precision    recall  f1-score   support

        Wake       0.85      0.91      0.88      1872
          N1       0.65      0.42      0.51       491
          N2       0.92      0.92      0.92      5546
          N3       0.91      0.91      0.91      1642
         REM       0.88      0.90      0.89      2029

    accuracy                           0.89     11580
   macro avg       0.84      0.81      0.82     11580
weighted avg       0.89      0.89      0.89     11580

Wake       | Sensitivity: 0.911 | Specificity: 0.970
N1         | Sensitivity: 0.418 | Specificity: 0.990
N2         | Sensitivity: 0.920 | Specificity: 0.928
N3         | Sensitivity: 0.911 | Specificity: 0.986
REM        | Sensitivity: 0.901 | Specificity: 0.973


====================== FINAL MODALITY COMPARISON ========================
------------------------------------------------------------------------------
Modality             |    Acc | F1-macro |  Kappa |
---------------

# RF

In [11]:
import h5py
import numpy as np
import lightgbm as lgb
import time

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score,
    confusion_matrix, classification_report,
    cohen_kappa_score
)
from sklearn.utils import resample
from sklearn.ensemble import RandomForestClassifier

# ==========================================================
# SELECT CLASSIFICATION MODE
# ==========================================================
classification_mode = "5class"   # "binary", "3class", "5class"


# ==========================================================
# 1. LOAD DATA
# ==========================================================
hdf5_path = r"C:\Users\anita\OneDrive - Universitetet i Oslo\Masteroppgave zzz\UOslo_March2025\Combined\Step4_normalized_train_test_FINAL.h5"

with h5py.File(hdf5_path, 'r') as f:
    X_train = f['X_train'][:]
    X_test = f['X_test'][:]
    y_train = f['y_train'][:]
    y_test = f['y_test'][:]
    feature_names = f['feature_names'][:].astype(str)

print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")


# ==========================================================
# 2. FILTER VALID LABELS
# ==========================================================
valid_labels = [0,1,2,3,5]

train_mask = np.isin(y_train, valid_labels)
test_mask  = np.isin(y_test, valid_labels)

X_train = X_train[train_mask]
y_train = y_train[train_mask]
X_test  = X_test[test_mask]
y_test  = y_test[test_mask]


# ==========================================================
# 3. LABEL MAPPING
# ==========================================================

if classification_mode == "binary":

    print("\nRunning Binary Sleep vs Wake")

    sleep_stages = [1,2,3,5]

    y_train_new = np.where(np.isin(y_train, sleep_stages),0,1)
    y_test_new  = np.where(np.isin(y_test, sleep_stages),0,1)

    target_names = ["Sleep","Wake"]


elif classification_mode == "3class":

    print("\nRunning Wake vs NREM vs REM")

    y_train_new = np.copy(y_train)
    y_test_new  = np.copy(y_test)

    y_train_new[np.isin(y_train,[1,2,3])] = 1
    y_test_new[np.isin(y_test,[1,2,3])] = 1

    y_train_new[y_train == 5] = 2
    y_test_new[y_test == 5] = 2

    target_names = ["Wake","NREM","REM"]


elif classification_mode == "5class":

    print("\nRunning Full 5-Stage Classification")

    y_train_new = np.copy(y_train)
    y_test_new  = np.copy(y_test)

    target_names = ["Wake","N1","N2","N3","REM"]


# ==========================================================
# 4. FEATURE GROUPING
# ==========================================================

EEG_features = [f for f in feature_names if f.startswith(("C3","C4","A1","A2"))]
EOG_features = [f for f in feature_names if f.startswith("EOG")]
EMG_features = [f for f in feature_names if f.startswith("EMG")]
cardiac = [f for f in feature_names if f.startswith(("ECG","Pleth","Pulse","SpO2","Thorax"))]
acc = [f for f in feature_names if "Accelerometer" in f]

feature_sets = {

    "EEG": EEG_features,
    "EEG + EOG": EEG_features + EOG_features,
    "EEG + EOG + EMG": EEG_features + EOG_features + EMG_features,
    "Cardiac": cardiac,
    "Accelerometer": acc,
    "Acc + EMG": acc + EMG_features,
    "Acc + Car": acc + cardiac,
    "Acc + Car + EMG": acc + cardiac + EMG_features,
    "All PSG features": EEG_features + EOG_features + EMG_features + cardiac,
    "All": list(feature_names)
}

feature_to_idx = {f:i for i,f in enumerate(feature_names)}


# ==========================================================
# 5. MODALITY LOOP
# ==========================================================

results = {}

for name, fs in feature_sets.items():

    print("\n=======================================")
    print(f"Training with modality: {name}")
    print(f"Number of features: {len(fs)}")

    if len(fs) == 0:
        continue

    idx = [feature_to_idx[f] for f in fs]

    X_train_fs = X_train[:, idx]
    X_test_fs = X_test[:, idx]


    # ======================================================
    # BALANCING
    # ======================================================

    if classification_mode != "5class":

        unique_classes = np.unique(y_train_new)
        max_samples = max([np.sum(y_train_new == c) for c in unique_classes])

        X_bal, y_bal = [], []

        for c in unique_classes:

            X_c = X_train_fs[y_train_new == c]
            y_c = y_train_new[y_train_new == c]

            if len(X_c) < max_samples:

                X_res, y_res = resample(
                    X_c, y_c,
                    replace=True,
                    n_samples=max_samples,
                    random_state=42
                )

            else:

                X_res, y_res = X_c, y_c

            X_bal.append(X_res)
            y_bal.append(y_res)

        X_train_final = np.vstack(X_bal)
        y_train_final = np.hstack(y_bal)

    else:

        X_train_final = X_train_fs
        y_train_final = y_train_new


    # ======================================================
    # SCALING
    # ======================================================

    scaler = StandardScaler()

    X_train_final = scaler.fit_transform(X_train_final)
    X_test_fs = scaler.transform(X_test_fs)


    # ======================================================
    # MODEL
    # ======================================================

    model = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
        )

    # ======================================================
    # TRAIN
    # ======================================================

    start = time.time()

    model.fit(X_train_final, y_train_final)

    print(f"Training time: {time.time()-start:.2f} sec")


    # ======================================================
    # PREDICT
    # ======================================================

    y_pred = model.predict(X_test_fs)


    # ======================================================
    # METRICS
    # ======================================================

    acc = accuracy_score(y_test_new, y_pred)
    f1 = f1_score(y_test_new, y_pred, average='macro')
    kappa = cohen_kappa_score(y_test_new, y_pred)

    cm = confusion_matrix(y_test_new, y_pred)

    #print("\nConfusion Matrix\n", cm)

    print("\nClassification Report\n")
    print(classification_report(y_test_new, y_pred, target_names=target_names))


    # ======================================================
    # STAGE-WISE SENSITIVITY & SPECIFICITY
    # ======================================================

    sensitivities = {}
    specificities = {}

    for i, label in enumerate(target_names):

        TP = cm[i,i]
        FN = np.sum(cm[i,:]) - TP
        FP = np.sum(cm[:,i]) - TP
        TN = np.sum(cm) - (TP + FN + FP)

        sens = TP / (TP + FN)
        spec = TN / (TN + FP)

        sensitivities[label] = sens
        specificities[label] = spec

        print(f"{label:10s} | Sensitivity: {sens:.3f} | Specificity: {spec:.3f}")


    results[name] = {
        "Accuracy": acc,
        "F1_macro": f1,
        "Kappa": kappa,
        "Sensitivity": sensitivities,
        "Specificity": specificities
    }


# ==========================================================
# 6. FINAL SUMMARY
# ==========================================================

print("\n\n====================== FINAL MODALITY COMPARISON ========================")
print("------------------------------------------------------------------------------")
print(f"{'Modality':20s} | {'Acc':>6s} | {'F1-macro':>8s} | {'Kappa':>6s} | {'Sens(W)':>8s} | {'Spec(S)':>8s}")
print("-------------------------------------------------------------------------------")

for k,v in results.items():

    print(f"{k:20s} | "
          f"{v['Accuracy']:6.3f} | "
          f"{v['F1_macro']:8.3f} | "
          f"{v['Kappa']:6.3f} | "
          f"{v['Sensitivity']['Wake']:8.3f} | "
          f"{v['Specificity']['Wake']:8.3f}")

print("--------------------------------------------------------------------------------")

Train shape: (46398, 129)
Test shape: (11600, 129)

Running Full 5-Stage Classification

Training with modality: EEG
Number of features: 60
Training time: 18.71 sec

Classification Report

              precision    recall  f1-score   support

        Wake       0.79      0.88      0.83      1872
          N1       0.53      0.29      0.37       491
          N2       0.90      0.88      0.89      5546
          N3       0.87      0.87      0.87      1642
         REM       0.79      0.84      0.81      2029

    accuracy                           0.84     11580
   macro avg       0.78      0.75      0.76     11580
weighted avg       0.84      0.84      0.84     11580

Wake       | Sensitivity: 0.880 | Specificity: 0.954
N1         | Sensitivity: 0.289 | Specificity: 0.989
N2         | Sensitivity: 0.877 | Specificity: 0.906
N3         | Sensitivity: 0.873 | Specificity: 0.979
REM        | Sensitivity: 0.837 | Specificity: 0.954

Training with modality: EEG + EOG
Number of features: 84

# SVM

In [13]:
import h5py
import numpy as np
import lightgbm as lgb
import time

from sklearn.calibration import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score,
    confusion_matrix, classification_report,
    cohen_kappa_score
)
from sklearn.utils import resample
from sklearn.svm import SVC

# ==========================================================
# SELECT CLASSIFICATION MODE
# ==========================================================
classification_mode = "5class"   # "binary", "3class", "5class"


# ==========================================================
# 1. LOAD DATA
# ==========================================================
hdf5_path = r"C:\Users\anita\OneDrive - Universitetet i Oslo\Masteroppgave zzz\UOslo_March2025\Combined\Step4_normalized_train_test_FINAL.h5"

with h5py.File(hdf5_path, 'r') as f:
    X_train = f['X_train'][:]
    X_test = f['X_test'][:]
    y_train = f['y_train'][:]
    y_test = f['y_test'][:]
    feature_names = f['feature_names'][:].astype(str)

print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")


# ==========================================================
# 2. FILTER VALID LABELS
# ==========================================================
valid_labels = [0,1,2,3,5]

train_mask = np.isin(y_train, valid_labels)
test_mask  = np.isin(y_test, valid_labels)

X_train = X_train[train_mask]
y_train = y_train[train_mask]
X_test  = X_test[test_mask]
y_test  = y_test[test_mask]


# ==========================================================
# 3. LABEL MAPPING
# ==========================================================

if classification_mode == "binary":

    print("\nRunning Binary Sleep vs Wake")

    sleep_stages = [1,2,3,5]

    y_train_new = np.where(np.isin(y_train, sleep_stages),0,1)
    y_test_new  = np.where(np.isin(y_test, sleep_stages),0,1)

    target_names = ["Sleep","Wake"]


elif classification_mode == "3class":

    print("\nRunning Wake vs NREM vs REM")

    y_train_new = np.copy(y_train)
    y_test_new  = np.copy(y_test)

    y_train_new[np.isin(y_train,[1,2,3])] = 1
    y_test_new[np.isin(y_test,[1,2,3])] = 1

    y_train_new[y_train == 5] = 2
    y_test_new[y_test == 5] = 2

    target_names = ["Wake","NREM","REM"]


elif classification_mode == "5class":

    print("\nRunning Full 5-Stage Classification")

    y_train_new = np.copy(y_train)
    y_test_new  = np.copy(y_test)

    target_names = ["Wake","N1","N2","N3","REM"]


# ==========================================================
# 4. FEATURE GROUPING
# ==========================================================

EEG_features = [f for f in feature_names if f.startswith(("C3","C4","A1","A2"))]
EOG_features = [f for f in feature_names if f.startswith("EOG")]
EMG_features = [f for f in feature_names if f.startswith("EMG")]
cardiac = [f for f in feature_names if f.startswith(("ECG","Pleth","Pulse","SpO2","Thorax"))]
acc = [f for f in feature_names if "Accelerometer" in f]

feature_sets = {

    "EEG": EEG_features,
    "EEG + EOG": EEG_features + EOG_features,
    "EEG + EOG + EMG": EEG_features + EOG_features + EMG_features,
    "Cardiac": cardiac,
    "Accelerometer": acc,
    "Acc + EMG": acc + EMG_features,
    "Acc + Car": acc + cardiac,
    "Acc + Car + EMG": acc + cardiac + EMG_features,
    "All PSG features": EEG_features + EOG_features + EMG_features + cardiac,
    "All": list(feature_names)
}

feature_to_idx = {f:i for i,f in enumerate(feature_names)}


# ==========================================================
# 5. MODALITY LOOP
# ==========================================================

results = {}

for name, fs in feature_sets.items():

    print("\n=======================================")
    print(f"Training with modality: {name}")
    print(f"Number of features: {len(fs)}")

    if len(fs) == 0:
        continue

    idx = [feature_to_idx[f] for f in fs]

    X_train_fs = X_train[:, idx]
    X_test_fs = X_test[:, idx]


    # ======================================================
    # BALANCING
    # ======================================================

    if classification_mode != "5class":

        unique_classes = np.unique(y_train_new)
        max_samples = max([np.sum(y_train_new == c) for c in unique_classes])

        X_bal, y_bal = [], []

        for c in unique_classes:

            X_c = X_train_fs[y_train_new == c]
            y_c = y_train_new[y_train_new == c]

            if len(X_c) < max_samples:

                X_res, y_res = resample(
                    X_c, y_c,
                    replace=True,
                    n_samples=max_samples,
                    random_state=42
                )

            else:

                X_res, y_res = X_c, y_c

            X_bal.append(X_res)
            y_bal.append(y_res)

        X_train_final = np.vstack(X_bal)
        y_train_final = np.hstack(y_bal)

    else:

        X_train_final = X_train_fs
        y_train_final = y_train_new


    # ======================================================
    # SCALING
    # ======================================================

    scaler = StandardScaler()

    X_train_final = scaler.fit_transform(X_train_final)
    X_test_fs = scaler.transform(X_test_fs)


    # ======================================================
    # MODEL
    # ======================================================

    svm_params = {
    "C": 1.0,
    "tol": 1e-4,
    "class_weight": "balanced",
    "max_iter": 5000
     }
    
    # ======================================================
    # TRAIN
    # ======================================================

    start = time.time()

    #model = LinearSVC(**svm_params)
    # model = SVC(
    # kernel="rbf",
    # C=10,
    # gamma="scale",
    # class_weight="balanced",
    # probability=False,
    # random_state=42
    
    
    model.fit(X_train_final, y_train_final)

    print(f"Training time: {time.time()-start:.2f} sec")

    # ======================================================
    # PREDICT
    # ======================================================

    y_pred = model.predict(X_test_fs)

    # ======================================================
    # METRICS
    # ======================================================

    acc = accuracy_score(y_test_new, y_pred)
    f1 = f1_score(y_test_new, y_pred, average='macro')
    kappa = cohen_kappa_score(y_test_new, y_pred)

    cm = confusion_matrix(y_test_new, y_pred)

    #print("\nConfusion Matrix\n", cm)

    print("\nClassification Report\n")
    print(classification_report(y_test_new, y_pred, target_names=target_names))


    # ======================================================
    # STAGE-WISE SENSITIVITY & SPECIFICITY
    # ======================================================

    sensitivities = {}
    specificities = {}

    for i, label in enumerate(target_names):

        TP = cm[i,i]
        FN = np.sum(cm[i,:]) - TP
        FP = np.sum(cm[:,i]) - TP
        TN = np.sum(cm) - (TP + FN + FP)

        sens = TP / (TP + FN)
        spec = TN / (TN + FP)

        sensitivities[label] = sens
        specificities[label] = spec

        print(f"{label:10s} | Sensitivity: {sens:.3f} | Specificity: {spec:.3f}")


    results[name] = {
        "Accuracy": acc,
        "F1_macro": f1,
        "Kappa": kappa,
        "Sensitivity": sensitivities,
        "Specificity": specificities
    }


# ==========================================================
# 6. FINAL SUMMARY
# ==========================================================

print("\n\n================= FINAL MODALITY COMPARISON ===============")
print("--------------------------------------------------------------------")
print(f"{'Modality':20s} | {'Acc':>6s} | {'F1-macro':>8s} | {'Kappa':>6s} |")
print("--------------------------------------------------------------------")

for k,v in results.items():

    print(f"{k:20s} | "
          f"{v['Accuracy']:6.3f} | "
          f"{v['F1_macro']:8.3f} | "
          f"{v['Kappa']:6.3f} | "
          f"{v['Sensitivity']['Wake']:8.3f} | "
          f"{v['Specificity']['Wake']:8.3f}")

print("----------------------------------------------------------------------")

Train shape: (46398, 129)
Test shape: (11600, 129)

Running Full 5-Stage Classification

Training with modality: EEG
Number of features: 60
Training time: 194.35 sec

Classification Report

              precision    recall  f1-score   support

        Wake       0.78      0.80      0.79      1872
          N1       0.27      0.58      0.37       491
          N2       0.93      0.78      0.85      5546
          N3       0.77      0.92      0.84      1642
         REM       0.79      0.76      0.78      2029

    accuracy                           0.79     11580
   macro avg       0.71      0.77      0.72     11580
weighted avg       0.83      0.79      0.80     11580

Wake       | Sensitivity: 0.802 | Specificity: 0.956
N1         | Sensitivity: 0.582 | Specificity: 0.929
N2         | Sensitivity: 0.776 | Specificity: 0.946
N3         | Sensitivity: 0.923 | Specificity: 0.954
REM        | Sensitivity: 0.764 | Specificity: 0.956

Training with modality: EEG + EOG
Number of features: 8

# Adaboost

In [14]:
import h5py
import numpy as np
import time

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score,
    confusion_matrix, classification_report,
    cohen_kappa_score
)

from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

from imblearn.over_sampling import SMOTE


# ==========================================================
# SELECT CLASSIFICATION MODE
# ==========================================================
classification_mode = "5class"   # "binary", "3class", "5class"


# ==========================================================
# 1. LOAD DATA
# ==========================================================
hdf5_path = r"C:\Users\anita\OneDrive - Universitetet i Oslo\Masteroppgave zzz\UOslo_March2025\Combined\Step4_normalized_train_test_FINAL.h5"

with h5py.File(hdf5_path, 'r') as f:
    X_train = f['X_train'][:]
    X_test = f['X_test'][:]
    y_train = f['y_train'][:]
    y_test = f['y_test'][:]
    feature_names = f['feature_names'][:].astype(str)

print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")


# ==========================================================
# 2. FILTER VALID LABELS
# ==========================================================
valid_labels = [0,1,2,3,5]

train_mask = np.isin(y_train, valid_labels)
test_mask  = np.isin(y_test, valid_labels)

X_train = X_train[train_mask]
y_train = y_train[train_mask]
X_test  = X_test[test_mask]
y_test  = y_test[test_mask]


# ==========================================================
# 3. LABEL MAPPING
# ==========================================================

if classification_mode == "binary":

    print("\nRunning Binary Sleep vs Wake")

    sleep_stages = [1,2,3,5]

    y_train_new = np.where(np.isin(y_train, sleep_stages),0,1)
    y_test_new  = np.where(np.isin(y_test, sleep_stages),0,1)

    target_names = ["Sleep","Wake"]


elif classification_mode == "3class":

    print("\nRunning Wake vs NREM vs REM")

    y_train_new = np.copy(y_train)
    y_test_new  = np.copy(y_test)

    y_train_new[np.isin(y_train,[1,2,3])] = 1
    y_test_new[np.isin(y_test,[1,2,3])] = 1

    y_train_new[y_train == 5] = 2
    y_test_new[y_test == 5] = 2

    target_names = ["Wake","NREM","REM"]


elif classification_mode == "5class":

    print("\nRunning Full 5-Stage Classification")

    y_train_new = np.copy(y_train)
    y_test_new  = np.copy(y_test)

    target_names = ["Wake","N1","N2","N3","REM"]


# ==========================================================
# 4. FEATURE GROUPING
# ==========================================================

EEG_features = [f for f in feature_names if f.startswith(("C3","C4","A1","A2"))]
EOG_features = [f for f in feature_names if f.startswith("EOG")]
EMG_features = [f for f in feature_names if f.startswith("EMG")]
cardiac = [f for f in feature_names if f.startswith(("ECG","Pleth","Pulse","SpO2","Thorax"))]
acc = [f for f in feature_names if "Accelerometer" in f]

feature_sets = {

    "EEG": EEG_features,
    "EEG + EOG": EEG_features + EOG_features,
    "EEG + EOG + EMG": EEG_features + EOG_features + EMG_features,
    "Cardiac": cardiac,
    "Accelerometer": acc,
    "Acc + EMG": acc + EMG_features,
    "Acc + Car": acc + cardiac,
    "Acc + Car + EMG": acc + cardiac + EMG_features,
    "All PSG features": EEG_features + EOG_features + EMG_features + cardiac,
    "All": list(feature_names)
}

feature_to_idx = {f:i for i,f in enumerate(feature_names)}



# ==========================================================
# 5. MODALITY LOOP
# ==========================================================

results = {}

for name, fs in feature_sets.items():

    print("\n=======================================")
    print(f"Training with modality: {name}")
    print(f"Number of features: {len(fs)}")

    if len(fs) == 0:
        continue

    idx = [feature_to_idx[f] for f in fs]

    X_train_fs = X_train[:, idx]
    X_test_fs = X_test[:, idx]


    # ======================================================
    # BALANCING (SMOTE)
    # ======================================================

    if classification_mode == "binary":

        smote = SMOTE(
            sampling_strategy=0.6,
            k_neighbors=3,
            random_state=42
        )

        X_train_final, y_train_final = smote.fit_resample(
            X_train_fs,
            y_train_new
        )

    elif classification_mode == "3class":

        smote = SMOTE(
            sampling_strategy="not majority",
            k_neighbors=3,
            random_state=42
        )

        X_train_final, y_train_final = smote.fit_resample(
            X_train_fs,
            y_train_new
        )

    else:

        X_train_final = X_train_fs
        y_train_final = y_train_new


    # ======================================================
    # SCALING
    # ======================================================

    scaler = StandardScaler()

    X_train_final = scaler.fit_transform(X_train_final)
    X_test_fs = scaler.transform(X_test_fs)


    # ======================================================
    # MODEL (AdaBoost)
    # ======================================================

    base_tree = DecisionTreeClassifier(
        max_depth=5,
        min_samples_leaf=10,
        random_state=42
    )

    model = AdaBoostClassifier(
        estimator=base_tree,
        n_estimators=50,
        learning_rate=0.1,
        random_state=42
    )


    # ======================================================
    # TRAIN
    # ======================================================

    start = time.time()

    model.fit(X_train_final, y_train_final)

    print(f"Training time: {time.time()-start:.2f} sec")


    # ======================================================
    # PREDICT
    # ======================================================

    y_pred = model.predict(X_test_fs)


    # ======================================================
    # METRICS
    # ======================================================

    acc = accuracy_score(y_test_new, y_pred)
    f1 = f1_score(y_test_new, y_pred, average='macro')
    kappa = cohen_kappa_score(y_test_new, y_pred)

    cm = confusion_matrix(y_test_new, y_pred)

    print("\nClassification Report\n")
    print(classification_report(y_test_new, y_pred, target_names=target_names))


    # ======================================================
    # STAGE-WISE SENSITIVITY & SPECIFICITY
    # ======================================================

    sensitivities = {}
    specificities = {}

    for i, label in enumerate(target_names):

        TP = cm[i,i]
        FN = np.sum(cm[i,:]) - TP
        FP = np.sum(cm[:,i]) - TP
        TN = np.sum(cm) - (TP + FN + FP)

        sens = TP / (TP + FN)
        spec = TN / (TN + FP)

        sensitivities[label] = sens
        specificities[label] = spec

        print(f"{label:10s} | Sensitivity: {sens:.3f} | Specificity: {spec:.3f}")


    results[name] = {
        "Accuracy": acc,
        "F1_macro": f1,
        "Kappa": kappa,
        "Sensitivity": sensitivities,
        "Specificity": specificities
    }


# ==========================================================
# 6. FINAL SUMMARY
# ==========================================================

print("\n\n=============== FINAL MODALITY COMPARISON ======================")
print("--------------------------------------------------------------------")
print(f"{'Modality':20s} | {'Acc':>6s} | {'F1-macro':>8s} | {'Kappa':>6s} |")
print("--------------------------------------------------------------------")

for k,v in results.items():

    print(f"{k:20s} | "
          f"{v['Accuracy']:6.3f} | "
          f"{v['F1_macro']:8.3f} | "
          f"{v['Kappa']:6.3f} | "
          f"{v['Sensitivity']['Wake']:8.3f} | "
          f"{v['Specificity']['Wake']:8.3f}")

print("---------------------------------------------------------------------")

Train shape: (46398, 129)
Test shape: (11600, 129)

Running Full 5-Stage Classification

Training with modality: EEG
Number of features: 60
Training time: 95.13 sec

Classification Report

              precision    recall  f1-score   support

        Wake       0.77      0.78      0.78      1872
          N1       0.37      0.02      0.04       491
          N2       0.83      0.87      0.85      5546
          N3       0.84      0.80      0.82      1642
         REM       0.68      0.76      0.72      2029

    accuracy                           0.79     11580
   macro avg       0.70      0.65      0.64     11580
weighted avg       0.78      0.79      0.78     11580

Wake       | Sensitivity: 0.781 | Specificity: 0.955
N1         | Sensitivity: 0.022 | Specificity: 0.998
N2         | Sensitivity: 0.868 | Specificity: 0.836
N3         | Sensitivity: 0.805 | Specificity: 0.974
REM        | Sensitivity: 0.764 | Specificity: 0.924

Training with modality: EEG + EOG
Number of features: 84

c:\Users\anita\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\anita\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\anita\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo

Training time: 16.57 sec

Classification Report

              precision    recall  f1-score   support

        Wake       0.71      0.56      0.63      1872
          N1       0.00      0.00      0.00       491
          N2       0.68      0.86      0.76      5546
          N3       0.79      0.65      0.71      1642
         REM       0.65      0.55      0.60      2029

    accuracy                           0.69     11580
   macro avg       0.57      0.52      0.54     11580
weighted avg       0.67      0.69      0.67     11580

Wake       | Sensitivity: 0.561 | Specificity: 0.956
N1         | Sensitivity: 0.000 | Specificity: 1.000
N2         | Sensitivity: 0.864 | Specificity: 0.626
N3         | Sensitivity: 0.646 | Specificity: 0.971
REM        | Sensitivity: 0.549 | Specificity: 0.938

Training with modality: Acc + Car
Number of features: 34


c:\Users\anita\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\anita\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\anita\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo

Training time: 33.54 sec

Classification Report

              precision    recall  f1-score   support

        Wake       0.69      0.69      0.69      1872
          N1       0.55      0.05      0.09       491
          N2       0.75      0.86      0.80      5546
          N3       0.83      0.70      0.76      1642
         REM       0.66      0.61      0.64      2029

    accuracy                           0.73     11580
   macro avg       0.70      0.58      0.60     11580
weighted avg       0.73      0.73      0.72     11580

Wake       | Sensitivity: 0.693 | Specificity: 0.941
N1         | Sensitivity: 0.047 | Specificity: 0.998
N2         | Sensitivity: 0.862 | Specificity: 0.731
N3         | Sensitivity: 0.699 | Specificity: 0.976
REM        | Sensitivity: 0.614 | Specificity: 0.934

Training with modality: Acc + Car + EMG
Number of features: 37
Training time: 37.60 sec

Classification Report

              precision    recall  f1-score   support

        Wake       0.72      

# CNN + LSTM

In [9]:
import h5py
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import time

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, cohen_kappa_score
)

from torch.utils.data import DataLoader, TensorDataset


# ==========================================================
# DEVICE
# ==========================================================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)


# ==========================================================
# LOAD DATA
# ==========================================================

hdf5_path = r"C:\Users\anita\OneDrive - Universitetet i Oslo\Masteroppgave zzz\UOslo_March2025\Combined\Step4_normalized_train_test_FINAL.h5"

with h5py.File(hdf5_path, 'r') as f:

    X_train = f['X_train'][:]
    X_test  = f['X_test'][:]

    y_train = f['y_train'][:]
    y_test  = f['y_test'][:]

    feature_names = f['feature_names'][:].astype(str)

print("Train shape:", X_train.shape)


# ==========================================================
# FILTER VALID LABELS
# ==========================================================

valid_labels = [0,1,2,3,5]

train_mask = np.isin(y_train, valid_labels)
test_mask  = np.isin(y_test, valid_labels)

X_train = X_train[train_mask]
X_test  = X_test[test_mask]

y_train = y_train[train_mask]
y_test  = y_test[test_mask]


# ==========================================================
# 5 CLASS MAPPING
# ==========================================================

label_map = {
    0:0,   # Wake
    1:1,   # N1
    2:2,   # N2
    3:3,   # N3
    5:4    # REM
}

y_train = np.vectorize(label_map.get)(y_train)
y_test  = np.vectorize(label_map.get)(y_test)

target_names = ["Wake","N1","N2","N3","REM"]


# ==========================================================
# FEATURE GROUPS
# ==========================================================

EEG_features = [f for f in feature_names if f.startswith(("C3","C4","A1","A2"))]
EOG_features = [f for f in feature_names if f.startswith("EOG")]
EMG_features = [f for f in feature_names if f.startswith("EMG")]
cardiac = [f for f in feature_names if f.startswith(("ECG","Pleth","Pulse","SpO2","Thorax"))]
acc = [f for f in feature_names if "Accelerometer" in f]

EEG_EOG = EEG_features + EOG_features
EEG_EOG_EMG = EEG_EOG + EMG_features

all_PSG_features = EEG_features + EOG_features + EMG_features + cardiac
all_features = list(feature_names)

feature_sets = {
    "EEG": EEG_features,
    "EEG + EOG": EEG_EOG,
    "EEG + EOG + EMG": EEG_EOG_EMG,
    "Cardiac": cardiac,
    "Accelerometer": acc,
    "Acc + EMG": acc + EMG_features,
    "Acc + Car": acc + cardiac,
    "Acc + Car + EMG": acc + cardiac + EMG_features,
    "All PSG features": all_PSG_features,
    "All": all_features
}

feature_to_idx = {f:i for i,f in enumerate(feature_names)}


# ==========================================================
# SEQUENCE FUNCTION
# ==========================================================

SEQ_LEN = 20

def build_sequences(X, y, seq_len=SEQ_LEN):

    X_seq, y_seq = [], []

    for i in range(len(X)-seq_len+1):

        X_seq.append(X[i:i+seq_len])
        y_seq.append(y[i+seq_len-1])

    return np.array(X_seq), np.array(y_seq)


# ==========================================================
# CNN LSTM MODEL
# ==========================================================

class CNN_LSTM(nn.Module):

    def __init__(self,n_features,hidden_size=128):

        super().__init__()

        self.conv1 = nn.Conv1d(1,32,kernel_size=5,padding=2)
        self.bn1   = nn.BatchNorm1d(32)

        self.conv2 = nn.Conv1d(32,64,kernel_size=5,padding=2)
        self.bn2   = nn.BatchNorm1d(64)

        self.pool = nn.MaxPool1d(2)
        self.relu = nn.ReLU()

        self.lstm = nn.LSTM(
            input_size=64,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.fc = nn.Linear(hidden_size*2,5)

    def forward(self,x):

        batch,seq_len,n_feat = x.size()

        x = x.view(batch*seq_len,1,n_feat)

        x = self.pool(self.relu(self.bn1(self.conv1(x))))
        x = self.pool(self.relu(self.bn2(self.conv2(x))))

        x = x.mean(dim=2)

        x = x.view(batch,seq_len,-1)

        out,_ = self.lstm(x)

        out = out[:,-1,:]

        return self.fc(out)


# ==========================================================
# MODALITY LOOP
# ==========================================================

results = {}

for name,fs in feature_sets.items():

    print("\n===================================")
    print("Training modality:", name)

    if len(fs)==0:
        continue

    idx = [feature_to_idx[f] for f in fs]

    X_train_fs = X_train[:,idx]
    X_test_fs  = X_test[:,idx]


    # ---------------- SCALING ----------------

    scaler = StandardScaler()

    X_train_fs = scaler.fit_transform(X_train_fs)
    X_test_fs  = scaler.transform(X_test_fs)


    # ---------------- SEQUENCES ----------------

    X_train_seq,y_train_seq = build_sequences(X_train_fs,y_train)
    X_test_seq,y_test_seq   = build_sequences(X_test_fs,y_test)


    # ---------------- TORCH ----------------

    X_train_seq = torch.tensor(X_train_seq,dtype=torch.float32).to(DEVICE)
    X_test_seq  = torch.tensor(X_test_seq,dtype=torch.float32).to(DEVICE)

    y_train_seq = torch.tensor(y_train_seq,dtype=torch.long).to(DEVICE)
    y_test_seq  = torch.tensor(y_test_seq,dtype=torch.long).to(DEVICE)


    train_ds = TensorDataset(X_train_seq,y_train_seq)

    train_loader = DataLoader(train_ds,batch_size=256,shuffle=True)


    # ---------------- MODEL ----------------

    model = CNN_LSTM(n_features=X_train_seq.shape[2]).to(DEVICE)

    from sklearn.utils.class_weight import compute_class_weight

    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=np.unique(y_train),
        y=y_train
    )

    class_weights = torch.tensor(class_weights,dtype=torch.float32).to(DEVICE)

    criterion = nn.CrossEntropyLoss(weight=class_weights)

    optimizer = optim.Adam(model.parameters(),lr=1e-4)


    # ---------------- TRAINING ----------------

    EPOCHS = 10

    start = time.time()

    for epoch in range(EPOCHS):

        model.train()

        total_loss = 0

        for xb,yb in train_loader:

            optimizer.zero_grad()

            logits = model(xb)

            loss = criterion(logits,yb)

            loss.backward()

            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}/{EPOCHS} Loss {total_loss/len(train_loader):.4f}")

    print("Training time:",time.time()-start)


    # =====================================================
    # EVALUATION
    # =====================================================

    model.eval()

    with torch.no_grad():

        logits = model(X_test_seq)

        preds = torch.argmax(logits,dim=1).cpu().numpy()

        y_true = y_test_seq.cpu().numpy()


    print("\nClassification Report\n")

    print(classification_report(y_true,preds,target_names=target_names))


    cm = confusion_matrix(y_true,preds)

    print("\nSensitivity / Specificity\n")

    for i,label in enumerate(target_names):

        TP = cm[i,i]
        FN = np.sum(cm[i,:]) - TP
        FP = np.sum(cm[:,i]) - TP
        TN = np.sum(cm) - (TP + FN + FP)

        sens = TP/(TP+FN)
        spec = TN/(TN+FP)

        print(f"{label:10s} | Sensitivity: {sens:.3f} | Specificity: {spec:.3f}")


    acc = accuracy_score(y_true,preds)
    f1 = f1_score(y_true,preds,average="macro")
    kappa = cohen_kappa_score(y_true,preds)


    results[name] = {
        "Accuracy":acc,
        "F1_macro":f1,
        "Kappa":kappa
    }


# ==========================================================
# FINAL SUMMARY
# ==========================================================

print("\n\n================= FINAL MODALITY COMPARISON ===============")
print("--------------------------------------------------------------------")
print(f"{'Modality':20s} | {'Acc':>6s} | {'F1-macro':>8s} | {'Kappa':>6s} |")
print("--------------------------------------------------------------------")

for k,v in results.items():

    print(f"{k:20s} | "
          f"{v['Accuracy']:6.3f} | "
          f"{v['F1_macro']:8.3f} | "
          f"{v['Kappa']:6.3f} |")

print("----------------------------------------------------------------------")

Using device: cpu
Train shape: (46398, 129)

Training modality: EEG
Epoch 1/10 Loss 1.4622
Epoch 2/10 Loss 1.0856
Epoch 3/10 Loss 0.9333
Epoch 4/10 Loss 0.8873
Epoch 5/10 Loss 0.8599
Epoch 6/10 Loss 0.8398
Epoch 7/10 Loss 0.8236
Epoch 8/10 Loss 0.8115
Epoch 9/10 Loss 0.8011
Epoch 10/10 Loss 0.7914
Training time: 650.7275342941284

Classification Report

              precision    recall  f1-score   support

        Wake       0.65      0.82      0.72      1871
          N1       0.19      0.49      0.27       490
          N2       0.89      0.71      0.79      5533
          N3       0.69      0.88      0.77      1640
         REM       0.76      0.54      0.63      2027

    accuracy                           0.71     11561
   macro avg       0.64      0.69      0.64     11561
weighted avg       0.77      0.71      0.73     11561


Sensitivity / Specificity

Wake       | Sensitivity: 0.819 | Specificity: 0.915
N1         | Sensitivity: 0.488 | Specificity: 0.906
N2         | Sensitiv

# CNN

In [1]:
import h5py
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import time

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, cohen_kappa_score
)

from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader, TensorDataset


# ==========================================================
# DEVICE
# ==========================================================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)


# ==========================================================
# TASK TYPE
# ==========================================================
# OPTIONS:
# "binary"
# "3stage"
# "5stage"

TASK = "5stage"


# ==========================================================
# LOAD DATA
# ==========================================================

hdf5_path = r"C:\Users\anita\OneDrive - Universitetet i Oslo\Masteroppgave zzz\UOslo_March2025\Combined\Step4_normalized_train_test_FINAL.h5"

with h5py.File(hdf5_path, 'r') as f:

    X_train = f['X_train'][:]
    X_test  = f['X_test'][:]

    y_train = f['y_train'][:]
    y_test  = f['y_test'][:]

    feature_names = f['feature_names'][:].astype(str)

print("Train shape:", X_train.shape)


# ==========================================================
# FILTER VALID LABELS
# ==========================================================

valid_labels = [0,1,2,3,5]

train_mask = np.isin(y_train, valid_labels)
test_mask  = np.isin(y_test, valid_labels)

X_train = X_train[train_mask]
X_test  = X_test[test_mask]

y_train = y_train[train_mask]
y_test  = y_test[test_mask]


# ==========================================================
# LABEL MAPPING
# ==========================================================

if TASK == "binary":

    sleep_stages = [1,2,3,5]

    y_train_bin = np.copy(y_train)
    y_test_bin  = np.copy(y_test)

    y_train_bin[np.isin(y_train_bin,sleep_stages)] = 0
    y_train_bin[y_train == 0] = 1

    y_test_bin[np.isin(y_test_bin,sleep_stages)] = 0
    y_test_bin[y_test == 0] = 1

    y_train = y_train_bin
    y_test  = y_test_bin

    target_names = ["Sleep","Wake"]
    N_CLASSES = 2


elif TASK == "3stage":

    label_map = {
        0:0,  # Wake
        1:1,  # NREM
        2:1,
        3:1,
        5:2   # REM
    }

    y_train = np.vectorize(label_map.get)(y_train)
    y_test  = np.vectorize(label_map.get)(y_test)

    target_names = ["Wake","NREM","REM"]
    N_CLASSES = 3


elif TASK == "5stage":

    label_map = {
        0:0,
        1:1,
        2:2,
        3:3,
        5:4
    }

    y_train = np.vectorize(label_map.get)(y_train)
    y_test  = np.vectorize(label_map.get)(y_test)

    target_names = ["Wake","N1","N2","N3","REM"]
    N_CLASSES = 5


# ==========================================================
# FEATURE GROUPS
# ==========================================================

EEG_features = [f for f in feature_names if f.startswith(("C3","C4","A1","A2"))]
EOG_features = [f for f in feature_names if f.startswith("EOG")]
EMG_features = [f for f in feature_names if f.startswith("EMG")]
cardiac = [f for f in feature_names if f.startswith(("ECG","Pleth","Pulse","SpO2","Thorax"))]
acc = [f for f in feature_names if "Accelerometer" in f]

EEG_EOG = EEG_features + EOG_features
EEG_EOG_EMG = EEG_EOG + EMG_features

all_PSG_features = EEG_features + EOG_features + EMG_features + cardiac
all_features = list(feature_names)

feature_sets = {

    "EEG": EEG_features,
    "EEG + EOG": EEG_EOG,
    "EEG + EOG + EMG": EEG_EOG_EMG,
    "Cardiac": cardiac,
    "Accelerometer": acc,
    "Acc + EMG": acc + EMG_features,
    "Acc + Car": acc + cardiac,
    "Acc + Car + EMG": acc + cardiac + EMG_features,
    "All PSG features": all_PSG_features,
    "All": all_features
}

feature_to_idx = {f:i for i,f in enumerate(feature_names)}


# ==========================================================
# SEQUENCE CREATION
# ==========================================================

SEQ_LEN = 20

def build_sequences(X,y):

    X_seq=[]
    y_seq=[]

    for i in range(len(X)-SEQ_LEN+1):

        X_seq.append(X[i:i+SEQ_LEN])
        y_seq.append(y[i+SEQ_LEN-1])

    return np.array(X_seq),np.array(y_seq)


# ==========================================================
# CNN MODEL
# ==========================================================

class CNN(nn.Module):

    def __init__(self,n_features,n_classes):

        super().__init__()

        self.conv1 = nn.Conv1d(n_features,64,kernel_size=5,padding=2)
        self.bn1 = nn.BatchNorm1d(64)

        self.conv2 = nn.Conv1d(64,128,kernel_size=5,padding=2)
        self.bn2 = nn.BatchNorm1d(128)

        self.pool = nn.MaxPool1d(2)
        self.relu = nn.ReLU()

        self.dropout = nn.Dropout(0.3)

        self.fc = nn.Linear(128,n_classes)


    def forward(self,x):

        x = x.permute(0,2,1)

        x = self.pool(self.relu(self.bn1(self.conv1(x))))
        x = self.pool(self.relu(self.bn2(self.conv2(x))))

        x = x.mean(dim=2)

        x = self.dropout(x)

        return self.fc(x)


# ==========================================================
# MODALITY LOOP
# ==========================================================

results = {}

for name,fs in feature_sets.items():

    print("\n===================================")
    print("Training modality:",name)

    if len(fs)==0:
        continue

    idx = [feature_to_idx[f] for f in fs]

    X_train_fs = X_train[:,idx]
    X_test_fs  = X_test[:,idx]


    # SCALING

    scaler = StandardScaler()

    X_train_fs = scaler.fit_transform(X_train_fs)
    X_test_fs  = scaler.transform(X_test_fs)


    # SEQUENCES

    X_train_seq,y_train_seq = build_sequences(X_train_fs,y_train)
    X_test_seq,y_test_seq   = build_sequences(X_test_fs,y_test)


    # TORCH

    X_train_seq = torch.tensor(X_train_seq,dtype=torch.float32).to(DEVICE)
    X_test_seq  = torch.tensor(X_test_seq,dtype=torch.float32).to(DEVICE)

    y_train_seq = torch.tensor(y_train_seq,dtype=torch.long).to(DEVICE)
    y_test_seq  = torch.tensor(y_test_seq,dtype=torch.long).to(DEVICE)


    train_ds = TensorDataset(X_train_seq,y_train_seq)
    train_loader = DataLoader(train_ds,batch_size=256,shuffle=True)


    # MODEL

    model = CNN(X_train_fs.shape[1],N_CLASSES).to(DEVICE)


    # CLASS WEIGHTS

    weights = compute_class_weight(
        class_weight="balanced",
        classes=np.unique(y_train_seq.cpu().numpy()),
        y=y_train_seq.cpu().numpy()
    )

    weights = torch.tensor(weights,dtype=torch.float32).to(DEVICE)

    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = optim.Adam(model.parameters(),lr=1e-4)


    # TRAINING

    EPOCHS = 10

    start = time.time()

    for epoch in range(EPOCHS):

        model.train()

        total_loss = 0

        for xb,yb in train_loader:

            optimizer.zero_grad()

            logits = model(xb)

            loss = criterion(logits,yb)

            loss.backward()

            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}/{EPOCHS} Loss {total_loss/len(train_loader):.4f}")

    print("Training time:",time.time()-start)


    # =====================================================
    # EVALUATION
    # =====================================================

    model.eval()

    with torch.no_grad():

        logits = model(X_test_seq)

        preds = torch.argmax(logits,dim=1).cpu().numpy()
        y_true = y_test_seq.cpu().numpy()

    
    print("\nClassification Report\n")
    print(classification_report(y_true,preds,target_names=target_names))

    cm= confusion_matrix(y_true,preds)
    print("\nSensitivity / Specificity\n")
    for i,label in enumerate(target_names):
        TP = cm[i,i]
        FN = np.sum(cm[i,:]) - TP
        FP = np.sum(cm[:,i]) - TP
        TN = np.sum(cm) - (TP + FN + FP)
        sens = TP/(TP+FN)
        spec = TN/(TN+FP)
        print(f"{label:10s} | Sensitivity: {sens:.3f} | Specificity: {spec:.3f}")


    acc = accuracy_score(y_true,preds)
    f1  = f1_score(y_true,preds,average="macro")
    kappa = cohen_kappa_score(y_true,preds)
    

    results[name] = {
        "Accuracy":acc,
        "F1_macro":f1,
        "Kappa":kappa
    }


# ==========================================================
# FINAL SUMMARY
# ==========================================================

print("\n\n================ FINAL MODALITY COMPARISON ===============")

print("--------------------------------------------------------------------")
print(f"{'Modality':20s} | {'Acc':>6s} | {'F1-macro':>8s} | {'Kappa':>6s}")
print("--------------------------------------------------------------------")

for k,v in results.items():

    print(f"{k:20s} | "
          f"{v['Accuracy']:6.3f} | "
          f"{v['F1_macro']:8.3f} | "
          f"{v['Kappa']:6.3f}")

print("--------------------------------------------------------------------")

Using device: cpu
Train shape: (46398, 129)

Training modality: EEG
Epoch 1/10 Loss 1.6160
Epoch 2/10 Loss 1.5514
Epoch 3/10 Loss 1.4832
Epoch 4/10 Loss 1.4103
Epoch 5/10 Loss 1.3349
Epoch 6/10 Loss 1.2628
Epoch 7/10 Loss 1.1926
Epoch 8/10 Loss 1.1401
Epoch 9/10 Loss 1.0949
Epoch 10/10 Loss 1.0559
Training time: 32.034778118133545

Classification Report

              precision    recall  f1-score   support

        Wake       0.63      0.68      0.66      1871
          N1       0.12      0.33      0.17       490
          N2       0.81      0.54      0.65      5533
          N3       0.56      0.86      0.68      1640
         REM       0.55      0.52      0.53      2027

    accuracy                           0.60     11561
   macro avg       0.53      0.59      0.54     11561
weighted avg       0.67      0.60      0.61     11561


Sensitivity / Specificity

Wake       | Sensitivity: 0.681 | Specificity: 0.924
N1         | Sensitivity: 0.335 | Specificity: 0.887
N2         | Sensiti

# LSTM

In [1]:
import h5py
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import time

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix,
    classification_report, cohen_kappa_score
)
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader, TensorDataset


# ==========================================================
# DEVICE
# ==========================================================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)


# ==========================================================
# SETTINGS
# ==========================================================

SEQ_LEN = 20
N_CLASSES = 5     # change to 5 for 5-stage classification


# ==========================================================
# 1. LOAD DATA
# ==========================================================

hdf5_path = r"C:\Users\anita\OneDrive - Universitetet i Oslo\Masteroppgave zzz\UOslo_March2025\Combined\Step4_normalized_train_test_FINAL.h5"

with h5py.File(hdf5_path, 'r') as f:

    X_train = f['X_train'][:]
    X_test  = f['X_test'][:]

    y_train = f['y_train'][:]
    y_test  = f['y_test'][:]

    feature_names = f['feature_names'][:].astype(str)

print("Train shape:", X_train.shape)


# ==========================================================
# 2. FILTER VALID LABELS
# ==========================================================

valid_labels = [0,1,2,3,5]

train_mask = np.isin(y_train, valid_labels)
test_mask  = np.isin(y_test, valid_labels)

X_train = X_train[train_mask]
X_test  = X_test[test_mask]

y_train = y_train[train_mask]
y_test  = y_test[test_mask]


# ==========================================================
# 3. LABEL MAPPING
# ==========================================================

def map_to_3stage(y):

    y_new = np.zeros_like(y)

    y_new[y == 0] = 0                # Wake
    y_new[np.isin(y,[1,2,5])] = 1    # Light
    y_new[y == 3] = 2                # Deep

    return y_new


def map_to_5stage(y):

    y_new = np.zeros_like(y)

    y_new[y == 0] = 0
    y_new[y == 1] = 1
    y_new[y == 2] = 2
    y_new[y == 3] = 3
    y_new[y == 5] = 4

    return y_new


if N_CLASSES == 3:

    y_train_labels = map_to_3stage(y_train)
    y_test_labels  = map_to_3stage(y_test)

    target_names = ["Wake","Light","Deep"]

elif N_CLASSES == 5:

    y_train_labels = map_to_5stage(y_train)
    y_test_labels  = map_to_5stage(y_test)

    target_names = ["Wake","N1","N2","N3","REM"]


# ==========================================================
# SEQUENCE CREATION
# ==========================================================

def build_sequences(X,y):

    X_seq=[]
    y_seq=[]

    for i in range(len(X)-SEQ_LEN+1):

        X_seq.append(X[i:i+SEQ_LEN])
        y_seq.append(y[i+SEQ_LEN-1])

    return np.array(X_seq),np.array(y_seq)


# ==========================================================
# FEATURE GROUPING
# ==========================================================

EEG_features = [f for f in feature_names if f.startswith(("C3","C4","A1","A2"))]
EOG_features = [f for f in feature_names if f.startswith("EOG")]
EMG_features = [f for f in feature_names if f.startswith("EMG")]
cardiac = [f for f in feature_names if f.startswith(("ECG","Pleth","Pulse","SpO2","Thorax"))]
acc = [f for f in feature_names if "Accelerometer" in f]

feature_sets = {

    "EEG": EEG_features,
    "EEG + EOG": EEG_features + EOG_features,
    "EEG + EOG + EMG": EEG_features + EOG_features + EMG_features,
    "Cardiac": cardiac,
    "Accelerometer": acc,
    "Acc + EMG": acc + EMG_features,
    "Acc + Car": acc + cardiac,
    "Acc + Car + EMG": acc + cardiac + EMG_features,
    "All PSG features": EEG_features + EOG_features + EMG_features + cardiac,
    "All": list(feature_names)
}

feature_to_idx = {f:i for i,f in enumerate(feature_names)}


# ==========================================================
# LSTM MODEL
# ==========================================================

class LSTMModel(nn.Module):

    def __init__(self,n_features,n_classes):

        super().__init__()

        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=128,
            num_layers=2,
            batch_first=True,
            dropout=0.3,
            bidirectional=True
        )

        self.fc = nn.Linear(256,n_classes)


    def forward(self,x):

        out,_ = self.lstm(x)

        out = out[:,-1,:]

        return self.fc(out)


# ==========================================================
# MODALITY LOOP
# ==========================================================

results = {}

for name,fs in feature_sets.items():

    print("\n=======================================")
    print("Training modality:",name)

    if len(fs)==0:
        continue

    idx = [feature_to_idx[f] for f in fs]

    X_train_fs = X_train[:,idx]
    X_test_fs  = X_test[:,idx]


    # ---------------- SCALING ----------------

    scaler = StandardScaler()

    X_train_fs = scaler.fit_transform(X_train_fs)
    X_test_fs  = scaler.transform(X_test_fs)


    # ---------------- SEQUENCES ----------------

    X_train_seq,y_train_seq = build_sequences(X_train_fs,y_train_labels)
    X_test_seq,y_test_seq   = build_sequences(X_test_fs,y_test_labels)


    # ---------------- TORCH ----------------

    X_train_t = torch.tensor(X_train_seq,dtype=torch.float32).to(DEVICE)
    X_test_t  = torch.tensor(X_test_seq,dtype=torch.float32).to(DEVICE)

    y_train_t = torch.tensor(y_train_seq,dtype=torch.long).to(DEVICE)
    y_test_t  = torch.tensor(y_test_seq,dtype=torch.long).to(DEVICE)


    train_ds = TensorDataset(X_train_t,y_train_t)

    train_loader = DataLoader(train_ds,batch_size=256,shuffle=True)


    # ---------------- MODEL ----------------

    model = LSTMModel(X_train_fs.shape[1],N_CLASSES).to(DEVICE)

    optimizer = optim.Adam(model.parameters(), lr=1e-3)


    # ---------------- CLASS WEIGHTS ----------------

    classes = np.arange(N_CLASSES)

    weights = compute_class_weight(
        class_weight='balanced',
        classes=classes,
        y=y_train_seq
    )

    print("Class weights:",weights)

    weights_tensor = torch.tensor(
        weights,
        dtype=torch.float32
    ).to(DEVICE)

    criterion = nn.CrossEntropyLoss(weight=weights_tensor)


    # ---------------- TRAIN ----------------

    EPOCHS = 10

    start = time.time()

    for epoch in range(EPOCHS):

        model.train()

        total_loss = 0

        for xb,yb in train_loader:

            optimizer.zero_grad()

            logits = model(xb)

            loss = criterion(logits,yb)

            loss.backward()

            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}/{EPOCHS} Loss {total_loss/len(train_loader):.4f}")

    print("Training time:",time.time()-start)


    # =====================================================
    # EVALUATION
    # =====================================================

    model.eval()

    with torch.no_grad():

        logits = model(X_test_t)

        preds = torch.argmax(logits,dim=1).cpu().numpy()

    y_true = y_test_seq


    print("\nClassification Report")
    print(classification_report(y_true,preds,target_names=target_names))


    cm = confusion_matrix(y_true,preds)

    print("\nSensitivity / Specificity\n")

    for i,label in enumerate(target_names):

        TP = cm[i,i]
        FN = np.sum(cm[i,:]) - TP
        FP = np.sum(cm[:,i]) - TP
        TN = np.sum(cm) - (TP + FN + FP)

        sens = TP/(TP+FN)
        spec = TN/(TN+FP)

        print(f"{label:10s} | Sensitivity: {sens:.3f} | Specificity: {spec:.3f}")


    acc = accuracy_score(y_true,preds)
    f1 = f1_score(y_true,preds,average="macro")
    kappa = cohen_kappa_score(y_true,preds)


    results[name] = {

        "Accuracy":acc,
        "F1":f1,
        "Kappa":kappa,
    }


# ==========================================================
# FINAL SUMMARY
# ==========================================================

print("\n\n====================== FINAL MODALITY COMPARISON ========================")

print("--------------------------------------------------------------------------")

print(f"{'Modality':20s} | {'Acc':>6s} | {'F1-macro':>8s} | {'Kappa':>6s}")

print("--------------------------------------------------------------------------")

for k,v in results.items():

    print(f"{k:20s} | "
          f"{v['Accuracy']:6.3f} | "
          f"{v['F1']:8.3f} | "
          f"{v['Kappa']:6.3f}")

print("--------------------------------------------------------------------------")

Using device: cpu
Train shape: (46398, 129)

Training modality: EEG
Class weights: [1.23739142 4.71456212 0.41754149 1.41063376 1.14172626]
Epoch 1/10 Loss 0.9594
Epoch 2/10 Loss 0.7575
Epoch 3/10 Loss 0.7184
Epoch 4/10 Loss 0.6892
Epoch 5/10 Loss 0.6671
Epoch 6/10 Loss 0.6408
Epoch 7/10 Loss 0.6191
Epoch 8/10 Loss 0.5943
Epoch 9/10 Loss 0.5622
Epoch 10/10 Loss 0.5299
Training time: 175.68760538101196

Classification Report
              precision    recall  f1-score   support

        Wake       0.74      0.78      0.76      1871
          N1       0.20      0.51      0.29       490
          N2       0.92      0.76      0.83      5533
          N3       0.77      0.90      0.83      1640
         REM       0.74      0.68      0.71      2027

    accuracy                           0.76     11561
   macro avg       0.67      0.73      0.68     11561
weighted avg       0.81      0.76      0.78     11561


Sensitivity / Specificity

Wake       | Sensitivity: 0.782 | Specificity: 0.947
N1